# Assignment 9 — Image Segmentation II

Manual quadtree region splitting, region adjacency merging, line detection, and gradient edge extraction.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LAB_DIR = Path.cwd()
def read_gray(name):
    image = cv2.imread(str(LAB_DIR / name), cv2.IMREAD_GRAYSCALE)
    if image is None: raise FileNotFoundError(LAB_DIR / name)
    return image

def split_regions(image, tolerance=40):
    regions = []
    def visit(r0, r1, c0, c1):
        block = image[r0:r1, c0:c1]
        if int(block.max()) - int(block.min()) <= tolerance or r1-r0 <= 1 or c1-c0 <= 1:
            regions.append((r0, r1, c0, c1)); return
        rm, cm = (r0+r1)//2, (c0+c1)//2
        for a, b, c, d in [(r0,rm,c0,cm),(r0,rm,cm,c1),(rm,r1,c0,cm),(rm,r1,cm,c1)]: visit(a,b,c,d)
    visit(0, image.shape[0], 0, image.shape[1]); return regions

def label_regions(shape, regions):
    labels = np.zeros(shape, np.uint8)
    for i, (r0,r1,c0,c1) in enumerate(regions, 1): labels[r0:r1,c0:c1] = i
    return labels

image = read_gray("simple_image.png")
regions = split_regions(image); labels = label_regions(image.shape, regions)
print("Regions after splitting:", len(regions)); plt.imshow(labels, cmap="nipy_spectral"); plt.title("Region splitting"); plt.axis("off"); plt.show()

parent = list(range(len(regions)))
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a,b):
    a,b=find(a),find(b)
    if a != b: parent[b]=a

def adjacent(a,b):
    r0,r1,c0,c1=regions[a]; s0,s1,d0,d1=regions[b]
    return (r1==s0 or s1==r0) and max(c0,d0)<min(c1,d1) or (c1==d0 or d1==c0) and max(r0,s0)<min(r1,s1)
for i in range(len(regions)):
    for j in range(i+1,len(regions)):
        if adjacent(i,j):
            a,b=regions[i],regions[j]; values=np.concatenate([image[a[0]:a[1],a[2]:a[3]].ravel(), image[b[0]:b[1],b[2]:b[3]].ravel()])
            if int(values.max())-int(values.min()) <= 40: union(i,j)
rag_labels = np.zeros_like(labels)
roots = {root: idx for idx, root in enumerate(sorted({find(i) for i in range(len(regions))}), 1)}
for i, region in enumerate(regions): rag_labels[region[0]:region[1],region[2]:region[3]] = roots[find(i)]
print("RAG nodes:", len(regions), "final regions:", len(roots)); plt.imshow(rag_labels, cmap="nipy_spectral"); plt.title("RAG result"); plt.axis("off"); plt.show()

def convolve(image, kernel, mode="constant"):
    p=kernel.shape[0]//2; padded=np.pad(image.astype(float),p,mode=mode); result=np.zeros(image.shape,float)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]): result[r,c]=(padded[r:r+2*p+1,c:c+2*p+1]*kernel).sum()
    return result

lines = read_gray("lines_w1.png")
kernels = [np.array([[-1,-1,-1],[2,2,2],[-1,-1,-1]]), np.array([[-1,-1,2],[-1,2,-1],[2,-1,-1]]), np.array([[-1,2,-1],[-1,2,-1],[-1,2,-1]]), np.array([[2,-1,-1],[-1,2,-1],[-1,-1,2]])]
responses = [convolve(lines, k) for k in kernels]; binary = [np.where(np.abs(r)>=100,255,0).astype(np.uint8) for r in responses]
fig, ax=plt.subplots(1,4,figsize=(14,4));
for a,b,title in zip(ax,binary,["H","+45","V","-45"]): a.imshow(b,cmap="gray"); a.set_title(title); a.axis("off")
plt.show()

peacock=read_gray("peacock.png"); smooth=convolve(peacock,np.ones((3,3))/9,"edge")
responses=[np.abs(convolve(smooth,k,"edge")) for k in kernels]; maximum=np.max(responses,axis=0); display=(255*maximum/(maximum.max() or 1)).astype(np.uint8); plt.imshow(display,cmap="gray"); plt.title("3-pixel line response"); plt.axis("off"); plt.show()

earth=cv2.resize(read_gray("earth.png"),(512,512)); gaussian=np.array([[2,4,5,4,2],[4,9,12,9,4],[5,12,15,12,5],[4,9,12,9,4],[2,4,5,4,2]],float)/159
smoothed=convolve(earth,gaussian); sx=np.array([[-1,0,1],[-2,0,2],[-1,0,1]]); sy=sx.T; gx=convolve(smoothed,sx); gy=convolve(smoothed,sy); magnitude=np.hypot(gx,gy); normalized=(255*(magnitude-magnitude.min())/(magnitude.ptp() or 1)).astype(np.uint8)
print(f"Gradient magnitude max: {magnitude.max():.3f}; 90th percentile: {np.percentile(magnitude,90):.3f}"); edges=np.where(normalized>=50,255,0).astype(np.uint8)
fig,ax=plt.subplots(1,3,figsize=(14,4));
for a,b,t in zip(ax,[smoothed,normalized,edges],["Gaussian", "Gradient magnitude", "Edges"]): a.imshow(b,cmap="gray"); a.set_title(t); a.axis("off")
plt.show()